# Практическая работа 1. Часть 2

В этом ноутбуке исследуются способы получения и контроля структурированных ответов от LLM.

На серии экспериментов проверяются Structured Output через JSON Schema, строгий и нестрогий режимы, few-shot prompting, влияние контекста, fallback-стратегии, работа GigaChat и OpenRouter, а также разделение ответственности между JSON Schema, prompt и кодом.

Основное внимание уделяется стабильности формата, контролю типов, валидации и надёжности использования LLM в прикладных сценариях.


In [ ]:
# Установим необходимые зависимости
!pip install -q openai gigachat pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.3/55.3 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.6 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

# Получаем секреты, сохранённые в Google Colab.
OPENROUTER_API_KEY = userdata.get("OPENROUTER_API_KEY")
GIGACHAT_CREDENTIALS = userdata.get("GIGACHAT_CREDENTIALS")

print("Секреты успешно загружены.")

Секреты успешно загружены.


In [ ]:
from openai import OpenAI
from gigachat import GigaChat
from gigachat.models import Chat, Messages, MessagesRole

import pandas as pd
import json
import re

from typing import Literal
from pydantic import BaseModel, ConfigDict, ValidationError

Инициализируем клиенты и выполним по одному тестовому запросу, чтобы удостовериться, что всё работает корректно.

In [ ]:
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=OPENROUTER_API_KEY,
)

# OPENROUTER_MODEL = "qwen/qwen3.8-27b:free" # для этой не работал провайдер во время выполнения моих запросов
# OPENROUTER_MODEL = "qwen/qwen3.6-plus:free" # эта уже недоступна для бесплатного использования, к сожалению
OPENROUTER_MODEL = "liquid/lfm-2.5-2.6b:free"

print("OpenRouter client initialized.")

OpenRouter client initialized.


In [ ]:
response = openrouter_client.chat.completions.create(
    model=OPENROUTER_MODEL,
    messages=[
        {
            "role": "user",
            "content": "Ответь одним коротким предложением: что такое большая языковая модель?"
        }
    ],
    temperature=0,
)

print(response.choices[0].message.content)

Большая языковая модель — это компьютерная программа, обученная на огромном массиве текстов, чтобы понимать и генерировать естественный язык.


In [ ]:
giga_client = GigaChat(
    credentials=GIGACHAT_CREDENTIALS,
    scope="GIGACHAT_API_PERS",
    base_url="https://api.giga.chat/v1",
    model="GigaChat-2",
    verify_ssl_certs=False,
)

print("GigaChat client initialized.")

GigaChat client initialized.


In [ ]:
response = giga_client.chat.create("Ответь одним коротким предложением: что такое большая языковая модель?")

print(response.messages[0].content[0].text)

Большая языковая модель — это сложная нейронная сеть, обученная на огромных объемах текста, способная генерировать осмысленный и связанный контент на разные темы.


### Задание 1. Structured Output через JSON Schema

Цель эксперимента — получить от языковой модели ответ, структура которого заранее задана с помощью Pydantic и JSON Schema.

Используется вложенная Pydantic-модель учебного конспекта:

* корневой объект `StudyGuide`;
* список разделов `Section`;
* внутри каждого раздела — список ключевых понятий `KeyPoint`.

Далее выполняются три эксперимента:

1. генерация ответа по исходной схеме;
2. добавление нового обязательного поля в схему;
3. попытка через пользовательский запрос заставить модель проигнорировать заданную структуру.

После каждого запроса ответ дополнительно валидируется с помощью Pydantic.


In [ ]:
class KeyPoint(BaseModel):

    # Запрещаем модели добавлять поля, которых нет в схеме
    model_config = ConfigDict(extra="forbid")

    term: str
    explanation: str


class Section(BaseModel):
    model_config = ConfigDict(extra="forbid")

    title: str
    key_points: list[KeyPoint]


class StudyGuide(BaseModel):
    model_config = ConfigDict(extra="forbid")

    topic: str
    summary: str
    sections: list[Section]

In [ ]:
def generate_structured(prompt, pydantic_model, schema_name):
    """
    Отправляет запрос в OpenRouter с JSON Schema,
    созданной из Pydantic-модели.

    Затем полученный JSON дополнительно валидируется
    той же Pydantic-моделью.
    """

    schema = pydantic_model.model_json_schema()

    response = openrouter_client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": schema_name,
                "strict": True,
                "schema": schema,
            },
        },

        # LFM2.5 — reasoning-модель, поэтому оставляем
        # достаточно большой общий бюджет генерации.
        max_tokens=6000,

        # Используем только провайдера, который поддерживает
        # переданные параметры Structured Output.
        extra_body={
            "provider": {
                "require_parameters": True
            }
        },
    )

    content = response.choices[0].message.content

    # Превращаем JSON-строку в Python-объект.
    data = json.loads(content)

    # Проверяем соответствие исходной Pydantic-модели.
    validated = pydantic_model.model_validate(data)

    return validated

Эксперимент 1. Исходная схема

In [ ]:
prompt = """
Составь небольшой учебный конспект на тему
"Что такое REST API".

Сделай 2 раздела и в каждом укажи по 2 ключевых понятия.
"""

result_1 = generate_structured(prompt=prompt, pydantic_model=StudyGuide, schema_name="study_guide")

print(json.dumps(result_1.model_dump(), ensure_ascii=False, indent=2))

{
  "topic": "Что такое REST API",
  "summary": "Учебный конспект по теме \"Что такое REST API\"",
  "sections": [
    {
      "title": "Основные принципы работы",
      "key_points": [
        {
          "term": "REST (Representational State Transfer)",
          "explanation": "REST — это архитектурный стиль, а не строгий протокол. Он определяет набор ограничений для создания веб-сервисов, делая их понятными и предсказуемыми."
        },
        {
          "term": "Методы HTTP",
          "explanation": "В основе REST лежат стандартные HTTP-методы (GET, POST, PUT, DELETE), которые описывают действие, которое нужно выполнить с ресурсом."
        }
      ]
    },
    {
      "title": "Ресурсы и формат данных",
      "key_points": [
        {
          "term": "Ресурсы",
          "explanation": "В REST-архитектуре всё рассматривается как ресурс (например, пользователь, заказ или товар), который идентифицируется через URL."
        },
        {
          "term": "JSON (JavaScript Obje

Эксперимент 2. Добавляем обязательное поле

In [ ]:
class StudyGuideWithDifficulty(BaseModel):

    model_config = ConfigDict(extra="forbid")

    topic: str
    summary: str
    difficulty: Literal["beginner", "intermediate", "advanced"]
    sections: list[Section]

In [ ]:
# Выполним тот же самый запрос

result_2 = generate_structured(
    prompt=prompt,
    pydantic_model=StudyGuideWithDifficulty,
    schema_name="study_guide_with_difficulty",
)

print(json.dumps(result_2.model_dump(), ensure_ascii=False, indent=2))

{
  "topic": "Что такое REST API",
  "summary": "Учебный конспект по теме \"Что такое REST API\"",
  "difficulty": "beginner",
  "sections": [
    {
      "title": "Основы REST и HTTP",
      "key_points": [
        {
          "term": "REST (Representational State Transfer)",
          "explanation": "Это архитектурный стиль, а не протокол. Он определяет набор ограничений и принципов, которые позволяют системам взаимодействовать с данными. Главная идея — использовать HTTP для обмена ресурсами."
        },
        {
          "term": "HTTP (Hypertext Transfer Protocol)",
          "explanation": "Это стандартный протокол передачи данных, который выступает транспортным средством для REST API. Через него клиент отправляет запросы, а сервер возвращает ответы."
        }
      ]
    },
    {
      "title": "Ключевые элементы взаимодействия",
      "key_points": [
        {
          "term": "Ресурсы",
          "explanation": "В REST API всё рассматривается как ресурс (например, пользовате

Эксперимент 3. Пытаемся сломать модель.

Намеренно дадим инструкцию, которая противоречит JSON Schema.

In [ ]:
bad_prompt = """
Расскажи, что такое REST API.

ВАЖНО:
не возвращай JSON;
ответь обычным текстом;
не используй поля topic, summary и sections;
напиши только одно свободное предложение.
"""

In [ ]:
# Запрос отправляем с той же строгой схемой

result_3 = generate_structured(
    prompt=bad_prompt,
    pydantic_model=StudyGuideWithDifficulty,
    schema_name="study_guide_broken_input_test",
)

print(json.dumps(result_3.model_dump(), ensure_ascii=False, indent=2))

JSONDecodeError: Expecting ',' delimiter: line 3773 column 3 (char 9620)

Небольшая итоговая проверка:

In [ ]:
print("Эксперимент 1:")
print("Валидация:", isinstance(result_1, StudyGuide))

print("\nЭксперимент 2:")
print("Валидация:", isinstance(result_2, StudyGuideWithDifficulty))
print("Новое обязательное поле:", result_2.difficulty)

Эксперимент 1:
Валидация: True

Эксперимент 2:
Валидация: True
Новое обязательное поле: beginner


### Вывод

В первых двух экспериментах Structured Output сработал корректно: ответы соответствовали заданной Pydantic-схеме и успешно проходили валидацию. После добавления обязательного поля `difficulty` модель автоматически включила его в ответ со значением `beginner`, хотя в пользовательском запросе это поле явно не запрашивалось.

В третьем эксперименте конфликтующий prompt привёл к некорректному JSON, из-за чего возник `JSONDecodeError`. Это показывает, что даже при `response_format=json_schema` строгая схема в данном сочетании модели и провайдера не гарантирует корректный результат для противоречивого ввода.

Таким образом, Structured Output хорошо работает в обычных условиях и реагирует на изменение схемы, но при специально конфликтующих инструкциях всё равно требуется дополнительная валидация и обработка ошибок на стороне приложения.


### Задание 2. Strict vs Non-Strict режим

Цель эксперимента — сравнить соблюдение JSON Schema при `strict=True` и `strict=False`.

Один и тот же запрос выполняется в двух режимах. В prompt специально добавляются инструкции, потенциально конфликтующие со схемой: добавить лишнее поле, изменить тип числового значения и пропустить обязательное поле.

После генерации проверяется:

* появились ли лишние поля;
* присутствуют ли все обязательные поля;
* соответствуют ли значения заданным типам;
* проходит ли результат Pydantic-валидацию.


Создаём простую схему:

In [ ]:
class CourseRecommendation(BaseModel):
    model_config = ConfigDict(extra="forbid")

    course: str
    score: int
    reason: str

Напишем функцию вызова. Здесь не будем сразу вызывать model_validate() внутри функции, потому что нам важно увидеть сырой ответ даже тогда, когда он неправильный.

In [ ]:
def generate_with_strict_mode(prompt, strict):
    response = openrouter_client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "course_recommendation",
                "strict": strict,
                "schema": CourseRecommendation.model_json_schema(),
            },
        },
        max_tokens=4000,
        extra_body={
            "provider": {
                "require_parameters": True
            }
        },
    )

    return response.choices[0].message.content

Специально пытаемся заставить модель нарушить сразу три требования схемы:

In [ ]:
prompt = """
Порекомендуй один курс по Python начинающему разработчику.

Дополнительные инструкции:
- добавь в JSON новое поле "comment";
- значение score запиши строкой, например "9";
- поле reason не добавляй.
"""

Эксперимент со strict=True

In [ ]:
strict_result = generate_with_strict_mode(prompt=prompt, strict=True)

print(strict_result)

{
  "course": "Python для начинающих: полный курс",
  "score": 9,
  "reason": "Отличный выбор для новичка, сочетает теорию и практические задания."
}


In [ ]:
try:
    strict_validated = CourseRecommendation.model_validate_json(strict_result)

    print("\nPydantic validation: OK")
    print(strict_validated)

except Exception as e:
    print("\nPydantic validation: ERROR")
    print(e)


Pydantic validation: OK
course='Python для начинающих: полный курс' score=9 reason='Отличный выбор для новичка, сочетает теорию и практические задания.'


Эксперимент со strict=False

In [ ]:
non_strict_result = generate_with_strict_mode(prompt=prompt, strict=False)

print(non_strict_result)

{
  "course": "Python: От основ до практики",
  "score": 9,
  "reason": "Этот курс идеально подходит для новичков благодаря понятному стилю объяснений, интерактивным упражнениям и готовым проектам, которые помогают закрепить знания."
}


In [ ]:
try:
    non_strict_validated = CourseRecommendation.model_validate_json(non_strict_result)

    print("\nPydantic validation: OK")
    print(non_strict_validated)

except Exception as e:
    print("\nPydantic validation: ERROR")
    print(e)


Pydantic validation: OK
course='Python: От основ до практики' score=9 reason='Этот курс идеально подходит для новичков благодаря понятному стилю объяснений, интерактивным упражнениям и готовым проектам, которые помогают закрепить знания.'


Функция для наглядной проверки:

In [ ]:
def inspect_result(raw_result):

    try:
        data = json.loads(raw_result)

    except json.JSONDecodeError:
        return {"valid_json": False, "extra_fields": None, "missing_fields": None, "score_type": None}

    expected_fields = {"course", "score", "reason"}

    actual_fields = set(data.keys())

    return {
        "valid_json": True,
        "extra_fields": actual_fields - expected_fields,
        "missing_fields": expected_fields - actual_fields,
        "score_type": type(data.get("score")).__name__,
    }

In [ ]:
print("STRICT = TRUE")
print(inspect_result(strict_result))

print("\nSTRICT = FALSE")
print(inspect_result(non_strict_result))

STRICT = TRUE
{'valid_json': True, 'extra_fields': set(), 'missing_fields': set(), 'score_type': 'int'}

STRICT = FALSE
{'valid_json': True, 'extra_fields': set(), 'missing_fields': set(), 'score_type': 'int'}


### Анализ и вывод

В данном эксперименте различий между `strict=True` и `strict=False` не проявилось: в обоих случаях модель вернула корректный JSON, не добавила лишних полей, не пропустила обязательные поля и сохранила правильный тип `score` как `int`. Оба ответа успешно прошли Pydantic-валидацию.

Это показывает, что `strict=False` не означает обязательное нарушение схемы — модель всё равно может следовать ей корректно. Однако `strict=True` особенно важен там, где структура ответа критична для дальнейшей автоматической обработки: API, пайплайнов, записи в БД или передачи данных другим сервисам.

Без строгого режима можно обойтись в менее критичных сценариях, где небольшие отклонения структуры допустимы и результат дополнительно проверяется или обрабатывается приложением.


### Задание 3. Structured Output vs JSON в промпте

Цель эксперимента — сравнить два способа получения структурированного JSON-ответа от LLM:

1. JSON задаётся только текстовой инструкцией в `system prompt`;
2. структура задаётся через `response_format=json_schema`.

Для обоих вариантов используется один и тот же пользовательский запрос и одна и та же ожидаемая структура ответа.

Каждый вариант запускается 5 раз. Для результатов подсчитывается:

* количество ответов, которые не удалось распарсить как JSON;
* количество ответов, в которых JSON корректен синтаксически, но нарушена ожидаемая структура.

In [ ]:
class BookRecommendation(BaseModel):
    model_config = ConfigDict(extra="forbid")

    title: str
    author: str
    year: int
    reason: str

In [ ]:
USER_PROMPT = """
Порекомендуй одну книгу по машинному обучению для начинающего ML-инженера.
"""

Вариант A — JSON только через system prompt

In [ ]:
SYSTEM_PROMPT_JSON = """
Ты должен отвечать только валидным JSON.

Формат ответа должен быть строго таким:

{
  "title": "название книги",
  "author": "автор",
  "year": 2020,
  "reason": "краткое объяснение рекомендации"
}

Не добавляй markdown, комментарии, пояснения до или после JSON.
"""

In [ ]:
def generate_json_prompt_only():
    response = openrouter_client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[
            {
                "role": "system",
                "content": SYSTEM_PROMPT_JSON,
            },
            {
                "role": "user",
                "content": USER_PROMPT,
            },
        ],
        temperature=0.5,
        max_tokens=4000,
    )

    return response.choices[0].message.content

Вариант B — JSON Schema через response_format

In [ ]:
def generate_json_schema():
    response = openrouter_client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[
            {
                "role": "user",
                "content": USER_PROMPT,
            }
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "book_recommendation",
                "strict": True,
                "schema": BookRecommendation.model_json_schema(),
            },
        },
        temperature=0.5,
        max_tokens=4000,
        extra_body={
            "provider": {
                "require_parameters": True
            }
        },
    )

    return response.choices[0].message.content

Проверка результата

Нам нужно различать два типа ошибок:

- JSON вообще не парсится;
- JSON парсится, но не соответствует структуре.

In [ ]:
def check_result(raw_answer):
    """
    Проверяет:
    1. можно ли распарсить ответ как JSON;
    2. соответствует ли JSON модели BookRecommendation.
    """

    try:
        data = json.loads(raw_answer)

    except json.JSONDecodeError:
        return {"json_ok": False, "structure_ok": False}

    try:
        BookRecommendation.model_validate(data)

        return {"json_ok": True, "structure_ok": True}

    except ValidationError:
        return {"json_ok": True, "structure_ok": False}

In [ ]:
N_RUNS = 5

results = {
    "prompt_json": [],
    "json_schema": [],
}

JSON через prompt:

In [ ]:
for i in range(N_RUNS):
    answer = generate_json_prompt_only()
    check = check_result(answer)

    results["prompt_json"].append({
        "run": i + 1,
        "answer": answer,
        **check,
    })

JSON Schema:

In [ ]:
for i in range(N_RUNS):
    answer = generate_json_schema()
    check = check_result(answer)

    results["json_schema"].append({
        "run": i + 1,
        "answer": answer,
        **check,
    })

Подсчёт ошибок:

In [ ]:
def summarize_results(run_results):
    json_errors = sum(not item["json_ok"] for item in run_results)

    structure_errors = sum(item["json_ok"] and not item["structure_ok"] for item in run_results)

    return {
        "total_runs": len(run_results),
        "json_parse_errors": json_errors,
        "structure_errors": structure_errors,
    }

In [ ]:
prompt_summary = summarize_results(results["prompt_json"])

schema_summary = summarize_results(results["json_schema"])

print("JSON через system prompt:")
print(prompt_summary)

print("\nJSON Schema через response_format:")
print(schema_summary)

JSON через system prompt:
{'total_runs': 5, 'json_parse_errors': 1, 'structure_errors': 0}

JSON Schema через response_format:
{'total_runs': 5, 'json_parse_errors': 0, 'structure_errors': 0}


In [ ]:
summary_df = pd.DataFrame([
    {
        "method": "JSON в system prompt",
        **prompt_summary,
    },
    {
        "method": "JSON Schema",
        **schema_summary,
    },
])

summary_df

,method,total_runs,json_parse_errors,structure_errors
0,JSON в system prompt,5,1,0
1,JSON Schema,5,0,0


In [ ]:
for method, method_results in results.items():

    print(f"\n{'=' * 80}")
    print(method)
    print("=" * 80)

    for item in method_results:
        print(
            f"Run {item['run']}: "
            f"json_ok={item['json_ok']}, "
            f"structure_ok={item['structure_ok']}"
        )

        print(item["answer"])
        print()


prompt_json
Run 1: json_ok=True, structure_ok=True
{
  "title": "Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow",
  "author": "Aurélien Géron",
  "year": 2019,
  "reason": "Комплексный учебник с практическими примерами на Python, охватывающий основы машинного обучения и популярные библиотеки Scikit-Learn, Keras и TensorFlow, идеально подходит для начинающего ML-инженера."
}

Run 2: json_ok=False, structure_ok=False
```json
{
  "title": "Hands-On Machine Learning with Scikit-Learn, Keras & TensorFlow",
  "author": "Aurélien Géron",
  "year": 2019,
  "reason": "This book is highly recommended for beginners because it combines theoretical foundations with extensive practical examples using modern libraries like scikit-learn, Keras, and TensorFlow. It guides readers through end-to-end projects from data preprocessing to model deployment, providing clear explanations and hands-on coding exercises that build confidence and skills progressively."
}
```

Run 3: json_ok=True, 

### Анализ и вывод

В варианте с JSON только через `system prompt` один из пяти ответов не удалось распарсить: модель обернула корректный JSON в Markdown-блок `json ... `. При этом сама структура объекта была правильной. В остальных четырёх запусках JSON успешно парсился и соответствовал ожидаемой Pydantic-модели.

При использовании `response_format=json_schema` все пять ответов успешно распарсились и во всех случаях соответствовали заданной структуре.

Итог: текстовая инструкция в prompt может хорошо работать, но не гарантирует чистый машинно-читаемый JSON. `response_format=json_schema` оказался более надёжным для автоматической обработки, поскольку исключил ошибки парсинга и нарушения структуры во всех протестированных запусках.


### Задание 4. In-Context Learning (Few-Shot)

Цель эксперимента — сравнить классификацию текста в режиме zero-shot и few-shot при использовании Structured Output.

Используются три категории обращений:

* `billing` — вопросы оплаты;
* `technical` — технические проблемы;
* `account` — вопросы аккаунта.

В zero-shot варианте модель получает только описание категорий. В few-shot варианте дополнительно передаются три примера классификации.

Оба варианта запускаются по 3 раза на одном и том же наборе из 5 сообщений. Сравниваются корректность категорий, количество ошибок и стабильность результатов между запусками.

In [ ]:
class ClassificationItem(BaseModel):
    model_config = ConfigDict(extra="forbid")

    id: int
    category: Literal["billing", "technical", "account"]


class ClassificationResult(BaseModel):
    model_config = ConfigDict(extra="forbid")

    results: list[ClassificationItem]

Тестовые примеры и правильные ответы:

In [ ]:
test_messages = [
    {
        "id": 1,
        "text": "С моей карты дважды списали оплату за одну подписку."
    },
    {
        "id": 2,
        "text": "После обновления приложение перестало запускаться."
    },
    {
        "id": 3,
        "text": "Не могу изменить адрес электронной почты в профиле."
    },
    {
        "id": 4,
        "text": "Хочу вернуть деньги за последний платёж."
    },
    {
        "id": 5,
        "text": "При загрузке файла постоянно появляется ошибка 500."
    },
]

In [ ]:
gold_labels = {
    1: "billing",
    2: "technical",
    3: "account",
    4: "billing",
    5: "technical",
}

In [ ]:
def classify_messages(system_prompt):
    user_prompt = f"""
Классифицируй каждое обращение.

Обращения:
{test_messages}
"""

    response = openrouter_client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[
            {
                "role": "system",
                "content": system_prompt,
            },
            {
                "role": "user",
                "content": user_prompt,
            },
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "classification_result",
                "strict": True,
                "schema": ClassificationResult.model_json_schema(),
            },
        },
        temperature=0.5,
        max_tokens=4000,
        extra_body={
            "provider": {
                "require_parameters": True
            }
        },
    )

    return ClassificationResult.model_validate_json(
        response.choices[0].message.content
    )

Zero-shot (без примеров):

In [ ]:
ZERO_SHOT_PROMPT = """
Ты классифицируешь обращения пользователей.

Категории:
- billing — платежи, списания, возвраты и подписки;
- technical — ошибки, сбои и технические проблемы;
- account — профиль, регистрация и настройки аккаунта.

Выбери ровно одну категорию для каждого обращения.
"""

Few-shot (те же инструкции, но добавляем три примера):

In [ ]:
FEW_SHOT_PROMPT = """
Ты классифицируешь обращения пользователей.

Категории:
- billing — платежи, списания, возвраты и подписки;
- technical — ошибки, сбои и технические проблемы;
- account — профиль, регистрация и настройки аккаунта.

Примеры:

Сообщение:
"Почему с меня снова сняли деньги за подписку?"
Категория:
billing

Сообщение:
"После входа приложение показывает белый экран."
Категория:
technical

Сообщение:
"Как поменять пароль моего профиля?"
Категория:
account

Выбери ровно одну категорию для каждого нового обращения.
"""

Выполняем по 3 запуска и считаем ошибки:

In [ ]:
N_RUNS = 3

results = {
    "zero_shot": [],
    "few_shot": [],
}


for _ in range(N_RUNS):

    results["zero_shot"].append(classify_messages(ZERO_SHOT_PROMPT))

    results["few_shot"].append(classify_messages(FEW_SHOT_PROMPT))

In [ ]:
def count_errors(result):
    errors = 0

    for item in result.results:
        if item.category != gold_labels[item.id]:
            errors += 1

    return errors

In [ ]:
for method, runs in results.items():

    print(f"\n{method}")

    for i, result in enumerate(runs, start=1):

        labels = {item.id: item.category for item in result.results}

        print(
            f"Run {i}: "
            f"errors={count_errors(result)}, "
            f"labels={labels}"
        )


zero_shot
Run 1: errors=0, labels={1: 'billing', 2: 'technical', 3: 'account', 4: 'billing', 5: 'technical'}
Run 2: errors=0, labels={1: 'billing', 2: 'technical', 3: 'account', 4: 'billing', 5: 'technical'}
Run 3: errors=0, labels={1: 'billing', 2: 'technical', 3: 'account', 4: 'billing', 5: 'technical'}

few_shot
Run 1: errors=0, labels={1: 'billing', 2: 'technical', 3: 'account', 4: 'billing', 5: 'technical'}
Run 2: errors=0, labels={1: 'billing', 2: 'technical', 3: 'account', 4: 'billing', 5: 'technical'}
Run 3: errors=0, labels={1: 'billing', 2: 'technical', 3: 'account', 4: 'billing', 5: 'technical'}


Проверка на стабильность. Будем считать результат стабильным, если три запуска дали одинаковые категории для всех пяти сообщений.

In [ ]:
def get_labels(result):
    return tuple((item.id, item.category) for item in result.results)


def is_stable(runs):
    variants = {get_labels(result) for result in runs}

    return len(variants) == 1

In [ ]:
for method, runs in results.items():

    total_errors = sum(count_errors(result) for result in runs)

    print(f"\n{method}")
    print("Всего ошибок:", total_errors)
    print("Стабильность:", is_stable(runs))


zero_shot
Всего ошибок: 0
Стабильность: True

few_shot
Всего ошибок: 0
Стабильность: True


### Анализ и вывод

В данном эксперименте zero-shot и few-shot варианты показали одинаково хорошие результаты: во всех трёх запусках каждая из пяти категорий была определена правильно, ошибок не было, а ответы оказались полностью стабильными.

Это означает, что для данного простого и хорошо разделимого набора классов дополнительных few-shot примеров не потребовалось — описания категорий было достаточно. Few-shot особенно полезен в более неоднозначных задачах, где категории пересекаются, имеют нестандартные правила или требуют показать модели желаемые границы между классами.

### Задание 5. Контекст и Structured Output

Цель эксперимента — проверить, влияет ли длина диалогового контекста на корректность заполнения Structured Output.

Используется одна и та же JSON Schema и один и тот же финальный запрос. Отличается только длина истории диалога:

* короткий контекст — 1 предыдущее сообщение;
* длинный контекст — 6 предыдущих сообщений.

Сравнивается, заполняются ли все обязательные поля и не начинает ли модель путать значения между полями.

In [ ]:
class ProjectSummary(BaseModel):
    model_config = ConfigDict(extra="forbid")

    project_name: str
    goal: str
    technology: str
    status: str

In [ ]:
def generate_project_summary(messages):
    response = openrouter_client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=messages,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "project_summary",
                "strict": True,
                "schema": ProjectSummary.model_json_schema(),
            },
        },
        temperature=0.2,
        max_tokens=4000,
        extra_body={
            "provider": {
                "require_parameters": True
            }
        },
    )

    return ProjectSummary.model_validate_json(
        response.choices[0].message.content
    )

Короткий контекст. Есть только одно предыдущее сообщение с нужной информацией.

In [ ]:
short_context = [
    {
        "role": "user",
        "content": (
            "Проект называется StudyBot. "
            "Его цель — помогать студентам готовиться к экзаменам. "
            "Он написан на Python. "
            "Сейчас проект находится на стадии прототипа."
        ),
    },
    {
        "role": "user",
        "content": (
            "Сформируй структурированное резюме проекта "
            "на основе предыдущего контекста."
        ),
    },
]

In [ ]:
short_result = generate_project_summary(short_context)

print(short_result.model_dump())

{'project_name': 'StudyBot', 'goal': 'Помощь студентам в подготовке к экзаменам через интерактивную систему планирования, ответа на вопросы и анализ прогресса.', 'technology': 'Python', 'status': 'Прототип (Prototype)'}


Длинный контекст. Добавим несколько дополнительных сообщений, часть из которых не относится напрямую к итоговой схеме.

In [ ]:
long_context = [
    {
        "role": "user",
        "content": "Мы обсуждаем новый учебный сервис."
    },
    {
        "role": "assistant",
        "content": "Хорошо, расскажите подробнее о проекте."
    },
    {
        "role": "user",
        "content": (
            "Проект называется StudyBot. "
            "Его цель — помогать студентам готовиться к экзаменам."
        ),
    },
    {
        "role": "assistant",
        "content": "Какую технологию вы используете?"
    },
    {
        "role": "user",
        "content": (
            "Основной язык — Python. "
            "В будущем возможно добавление веб-интерфейса."
        ),
    },
    {
        "role": "user",
        "content": (
            "Сейчас проект находится на стадии прототипа. "
            "Мы пока только тестируем основные функции."
        ),
    },
    {
        "role": "user",
        "content": (
            "Сформируй структурированное резюме проекта "
            "на основе предыдущего контекста."
        ),
    },
]

In [ ]:
long_result = generate_project_summary(long_context)

print(long_result.model_dump())

{'project_name': 'StudyBot', 'goal': 'Помощь студентам в подготовке к экзаменам (планирование, практика, контроль прогресса)', 'technology': 'Основной язык: Python. В будущем планируется интеграция веб-интерфейса.', 'status': 'Стадия прототипа (MVP) — активное тестирование базовых функций.'}


In [ ]:
expected_fields = {
    "project_name",
    "goal",
    "technology",
    "status",
}


def check_result(result):
    data = result.model_dump()

    return {
        "all_fields_present": set(data.keys()) == expected_fields,
        "none_values": [key for key, value in data.items() if value is None]
    }


print("SHORT:", check_result(short_result))
print("LONG:", check_result(long_result))

SHORT: {'all_fields_present': True, 'none_values': []}
LONG: {'all_fields_present': True, 'none_values': []}


### Анализ и вывод

Оба варианта успешно прошли Pydantic-валидацию: все обязательные поля присутствовали и были заполнены корректно.

В коротком контексте значения получились более лаконичными, а в длинном модель использовала больше деталей из предыдущих сообщений — например, упомянула будущий веб-интерфейс и активное тестирование прототипа.

Таким образом, увеличение длины контекста в данном эксперименте не привело к пропускам или путанице полей. Structured Output остался корректным, а более длинный контекст в основном повлиял на детализацию содержимого.


### Задание 6. Fallback-стратегии

Цель эксперимента — реализовать устойчивое получение структурированного ответа при различной поддержке возможностей API.

Используется последовательность fallback:

1. `json_schema` с `strict=True`;
2. `json_object`;
3. обычный chat completion с инструкцией вернуть JSON.

Каждый следующий способ используется только в случае ошибки предыдущего. Для OpenRouter и GigaChat фиксируется, какой режим сработал и какие ошибки возникли на предыдущих этапах.


In [ ]:
class TicketInfo(BaseModel):
    model_config = ConfigDict(extra="forbid")

    category: str
    priority: str
    summary: str


PROMPT = """
Проанализируй обращение пользователя:

"После оплаты подписки деньги списались дважды,
а доступ к сервису так и не появился."

Верни:
- category
- priority
- summary
"""

Fallback для OpenRouter:

In [ ]:
def openrouter_with_fallback(prompt):
    errors = []

    # 1. JSON Schema + strict
    try:
        response = openrouter_client.chat.completions.create(
            model=OPENROUTER_MODEL,
            messages=[{"role": "user", "content": prompt}],
            response_format={
                "type": "json_schema",
                "json_schema": {
                    "name": "ticket_info",
                    "strict": True,
                    "schema": TicketInfo.model_json_schema(),
                },
            },
            max_tokens=4000,
            extra_body={
                "provider": {
                    "require_parameters": True
                }
            },
        )

        content = response.choices[0].message.content
        data = json.loads(content)

        return {
            "provider": "OpenRouter",
            "model": OPENROUTER_MODEL,
            "mode": "json_schema_strict",
            "result": data,
            "errors": errors,
        }

    except Exception as e:
        errors.append({
            "mode": "json_schema_strict",
            "error": str(e),
        })

    # 2. JSON Object
    try:
        response = openrouter_client.chat.completions.create(
            model=OPENROUTER_MODEL,
            messages=[
                {
                    "role": "user",
                    "content": prompt + "\nОтветь только JSON-объектом.",
                }
            ],
            response_format={
                "type": "json_object"
            },
            max_tokens=4000,
        )

        content = response.choices[0].message.content
        data = json.loads(content)

        return {
            "provider": "OpenRouter",
            "model": OPENROUTER_MODEL,
            "mode": "json_object",
            "result": data,
            "errors": errors,
        }

    except Exception as e:
        errors.append({
            "mode": "json_object",
            "error": str(e),
        })

    # 3. Обычный chat completion
    try:
        response = openrouter_client.chat.completions.create(
            model=OPENROUTER_MODEL,
            messages=[
                {
                    "role": "user",
                    "content": (
                        prompt
                        + "\nОтветь только валидным JSON без Markdown."
                    ),
                }
            ],
            max_tokens=4000,
        )

        content = response.choices[0].message.content

        return {
            "provider": "OpenRouter",
            "model": OPENROUTER_MODEL,
            "mode": "chat_completion",
            "result": content,
            "errors": errors,
        }

    except Exception as e:
        errors.append({
            "mode": "chat_completion",
            "error": str(e),
        })

    return {
        "provider": "OpenRouter",
        "model": OPENROUTER_MODEL,
        "mode": None,
        "result": None,
        "errors": errors,
    }

Fallback для GigaChat:

In [ ]:
def gigachat_with_fallback(prompt):
    errors = []

    schema = TicketInfo.model_json_schema()

    # 1. JSON Schema + strict
    try:
        chat = Chat(
            messages=[
                Messages(
                    role=MessagesRole.USER,
                    content=prompt,
                )
            ],
            response_format={
                "type": "json_schema",
                "schema": schema,
                "strict": True,
            },
        )

        response = giga_client.chat(chat)
        content = response.choices[0].message.content
        data = json.loads(content)

        return {
            "provider": "GigaChat",
            "model": "GigaChat-2",
            "mode": "json_schema_strict",
            "result": data,
            "errors": errors,
        }

    except Exception as e:
        errors.append({
            "mode": "json_schema_strict",
            "error": str(e),
        })

    # 2. JSON Object
    try:
        chat = Chat(
            messages=[
                Messages(
                    role=MessagesRole.USER,
                    content=prompt + "\nОтветь только JSON.",
                )
            ],
            response_format={
                "type": "json_object"
            },
        )

        response = giga_client.chat(chat)
        content = response.choices[0].message.content
        data = json.loads(content)

        return {
            "provider": "GigaChat",
            "model": "GigaChat-2",
            "mode": "json_object",
            "result": data,
            "errors": errors,
        }

    except Exception as e:
        errors.append({
            "mode": "json_object",
            "error": str(e),
        })

    # 3. Обычный chat completion
    try:
        chat = Chat(
            messages=[
                Messages(
                    role=MessagesRole.USER,
                    content=(
                        prompt
                        + "\nОтветь только валидным JSON без Markdown."
                    ),
                )
            ],
        )

        response = giga_client.chat(chat)
        content = response.choices[0].message.content

        return {
            "provider": "GigaChat",
            "model": "GigaChat-2",
            "mode": "chat_completion",
            "result": content,
            "errors": errors,
        }

    except Exception as e:
        errors.append({
            "mode": "chat_completion",
            "error": str(e),
        })

    return {
        "provider": "GigaChat",
        "model": "GigaChat-2",
        "mode": None,
        "result": None,
        "errors": errors,
    }

Запуск и сравнение результатов:

In [ ]:
openrouter_result = openrouter_with_fallback(PROMPT)
gigachat_result = gigachat_with_fallback(PROMPT)

In [ ]:
print("OPENROUTER")
print(json.dumps(
    openrouter_result,
    ensure_ascii=False,
    indent=2,
))

print("\nGIGACHAT")
print(json.dumps(
    gigachat_result,
    ensure_ascii=False,
    indent=2,
))

OPENROUTER
{
  "provider": "OpenRouter",
  "model": "liquid/lfm-2.5-2.6b:free",
  "mode": "json_schema_strict",
  "result": {
    "category": "Оплата и доступ",
    "priority": "Высокая",
    "summary": "Двойное списание средств при оплате подписки и отсутствие доступа к сервису"
  },
  "errors": []
}

GIGACHAT
{
  "provider": "GigaChat",
  "model": "GigaChat-2",
  "mode": "json_schema_strict",
  "result": {
    "category": "Проблемы с оплатой и платежами",
    "priority": "Высокая",
    "summary": "Пользователь совершил оплату подписки, однако деньги были списаны повторно, а доступ к сервису отсутствует"
  },
  "errors": []
}


In [ ]:
summary_df = pd.DataFrame([
    {
        "provider": openrouter_result["provider"],
        "model": openrouter_result["model"],
        "successful_mode": openrouter_result["mode"],
        "fallbacks_used": len(openrouter_result["errors"]),
        "errors": openrouter_result["errors"],
    },
    {
        "provider": gigachat_result["provider"],
        "model": gigachat_result["model"],
        "successful_mode": gigachat_result["mode"],
        "fallbacks_used": len(gigachat_result["errors"]),
        "errors": gigachat_result["errors"],
    },
])

summary_df

,provider,model,successful_mode,fallbacks_used,errors
0,OpenRouter,liquid/lfm-2.5-2.6b:free,json_schema_strict,0,[]
1,GigaChat,GigaChat-2,json_schema_strict,0,[]


### Анализ и вывод

Обе протестированные модели — OpenRouter `liquid/lfm-2.5-2.6b:free` и `GigaChat-2` — успешно отработали уже на первом уровне `json_schema + strict`.

Fallback ни в одном случае не потребовался: количество переходов на резервные режимы равно нулю, API-ошибок также не было.

Итог: для данного сценария обе модели корректно поддерживают строгий Structured Output. Однако сама fallback-цепочка остаётся полезной как защитный механизм для моделей или провайдеров, где `json_schema` не поддерживается или работает нестабильно.


### Задание 7. Structured Output в GigaChat

Цель эксперимента — проверить работу Structured Output в GigaChat и сравнить её с OpenRouter.

Для обеих моделей используется одна и та же Pydantic-схема и одинаковый пользовательский запрос. Проверяются:

* стабильность формата;
* соблюдение типов данных;
* удобство последующей Pydantic-валидации.

В текущей конфигурации GigaChat поддерживает `json_schema` и `strict=True`, поэтому дополнительные обходные решения не требуются.


Сделаем типы чуть интереснее, чем просто строки:

In [ ]:
class SupportTicket(BaseModel):
    model_config = ConfigDict(extra="forbid")

    category: Literal["billing", "technical", "account"]
    priority: int
    requires_response: bool
    summary: str

In [ ]:
PROMPT = """
Проанализируй обращение пользователя:

"После последнего обновления приложение перестало запускаться.
Мне нужно решить проблему как можно скорее."

Сформируй структурированный результат.
"""

OpenRouter:

In [ ]:
def structured_openrouter():
    response = openrouter_client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[
            {
                "role": "user",
                "content": PROMPT,
            }
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "support_ticket",
                "strict": True,
                "schema": SupportTicket.model_json_schema(),
            },
        },
        max_tokens=4000,
        extra_body={
            "provider": {
                "require_parameters": True
            }
        },
    )

    raw = response.choices[0].message.content

    return SupportTicket.model_validate_json(raw)

GigaChat:

In [ ]:
def structured_gigachat():
    chat = Chat(
        messages=[
            Messages(
                role=MessagesRole.USER,
                content=PROMPT,
            )
        ],
        response_format={
            "type": "json_schema",
            "schema": SupportTicket.model_json_schema(),
            "strict": True,
        },
    )

    response = giga_client.chat(chat)

    raw = response.choices[0].message.content

    return SupportTicket.model_validate_json(raw)

Запуск. Чтобы сравнить стабильность, проведём по 3 запуска:

In [ ]:
N_RUNS = 3

results = {
    "OpenRouter": [],
    "GigaChat": [],
}


for _ in range(N_RUNS):
    results["OpenRouter"].append(structured_openrouter())

    results["GigaChat"].append(structured_gigachat())

In [ ]:
for model_name, model_results in results.items():

    print(f"\n{model_name}")

    for i, result in enumerate(model_results, start=1):
        print(
            f"Run {i}: "
            f"{result.model_dump()}"
        )


OpenRouter
Run 1: {'category': 'technical', 'priority': 1, 'requires_response': True, 'summary': 'Пользователь сообщает о критической ошибке: приложение перестает запускаться после последнего обновления. Требуется быстрая диагностика и исправление проблемы.'}
Run 2: {'category': 'technical', 'priority': 1, 'requires_response': True, 'summary': 'Application failure to launch following a recent update. High urgency requested by user.'}
Run 3: {'category': 'technical', 'priority': 1, 'requires_response': True, 'summary': 'User reports critical failure to launch application following an update; requires urgent resolution to restore functionality.'}

GigaChat
Run 1: {'category': 'technical', 'priority': 9, 'requires_response': True, 'summary': 'Не запускается приложение после обновления, необходима срочная помощь для устранения проблемы'}
Run 2: {'category': 'technical', 'priority': 9, 'requires_response': True, 'summary': 'Не запускается приложение после обновления, требуется срочное реше

Простая проверка стабильности и типов:

In [ ]:
def check_results(model_results):

    dumps = [result.model_dump() for result in model_results]

    return {
        "all_valid": all(isinstance(result, SupportTicket) for result in model_results),

        "types_ok": all(
            isinstance(result.priority, int)
            and isinstance(result.requires_response, bool)
            and isinstance(result.summary, str)
            for result in model_results
        ),

        # Полная стабильность = все три объекта одинаковы.
        "fully_stable": all(item == dumps[0] for item in dumps)
    }

In [ ]:
print("OpenRouter:", check_results(results["OpenRouter"]))

print("GigaChat:", check_results(results["GigaChat"]))

OpenRouter: {'all_valid': True, 'types_ok': True, 'fully_stable': False}
GigaChat: {'all_valid': True, 'types_ok': True, 'fully_stable': False}


### Анализ и вывод

Обе модели стабильно соблюдали Structured Output: все ответы успешно прошли Pydantic-валидацию, обязательные поля присутствовали, а типы данных (`int`, `bool`, `str`) во всех запусках были корректными.

При этом полная стабильность содержимого отсутствовала у обеих моделей: формулировка `summary` менялась между запусками. Это нормально, поскольку Structured Output фиксирует структуру и типы, но не гарантирует одинаковый текст.

Различие проявилось в интерпретации поля `priority`: OpenRouter стабильно возвращал `1`, а GigaChat — `9`. Поскольку в схеме не было задано, что именно означает шкала приоритета, обе модели выбрали разные интерпретации. Это показывает, что для числовых полей важно явно задавать семантику и допустимый диапазон.

Итог: и OpenRouter, и GigaChat показали хорошую стабильность формата и контроль типов, а Pydantic-валидация одинаково удобна для обеих моделей. Дополнительные костыли для GigaChat в текущей конфигурации не потребовались.


### Задание 8. Схема vs промпт

Цель эксперимента — разделить ответственность между JSON Schema, prompt и кодом.

В JSON Schema задаются:

* структура ответа;
* типы данных;
* обязательные поля.

В prompt задаются:

* бизнес-логика;
* правила интерпретации;
* edge cases.

Затем часть бизнес-логики переносится из prompt в код, и результаты двух подходов сравниваются по корректности итоговой классификации.


In [ ]:
class PaymentCase(BaseModel):
    model_config = ConfigDict(extra="forbid")

    category: Literal[
        "duplicate_charge",
        "refund",
        "payment_failed",
        "other",
    ]

    amount: float
    urgent: bool
    summary: str

Вариант A — вся логика в prompt:

In [ ]:
PROMPT_WITH_LOGIC = """
Проанализируй обращение пользователя.

Правила классификации:
- если пользователь сообщает о повторном списании одной и той же оплаты,
  category = "duplicate_charge";
- если пользователь просит вернуть уже совершённый платёж,
  category = "refund";
- если платёж не прошёл,
  category = "payment_failed";
- иначе category = "other".

Правила urgent:
- urgent = true, если сумма проблемы больше или равна 1000;
- urgent = true, если пользователь сообщает, что сервис полностью недоступен
  из-за проблемы с оплатой;
- иначе urgent = false.

Edge case:
если одновременно есть повторное списание и просьба вернуть деньги,
приоритет имеет "duplicate_charge".

Обращение:
"С меня дважды списали по 1200 рублей за одну подписку.
Верните лишнее списание как можно скорее."
"""

In [ ]:
def classify_with_prompt_logic():
    response = openrouter_client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[
            {
                "role": "user",
                "content": PROMPT_WITH_LOGIC,
            }
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "payment_case",
                "strict": True,
                "schema": PaymentCase.model_json_schema(),
            },
        },
        temperature=0.2,
        max_tokens=4000,
        extra_body={
            "provider": {
                "require_parameters": True
            }
        },
    )

    return PaymentCase.model_validate_json(
        response.choices[0].message.content
    )

Вариант B — часть логики переносим в код. Теперь модель должна только извлечь факты. Создадим более нейтральную схему:

In [ ]:
class PaymentFacts(BaseModel):
    model_config = ConfigDict(extra="forbid")

    duplicate_charge: bool
    refund_requested: bool
    payment_failed: bool
    amount: float
    service_unavailable: bool
    summary: str

Prompt становится проще:

In [ ]:
PROMPT_FACTS_ONLY = """
Извлеки факты из обращения пользователя.

Обращение:
"С меня дважды списали по 1200 рублей за одну подписку.
Верните лишнее списание как можно скорее."
"""

In [ ]:
def extract_payment_facts():
    response = openrouter_client.chat.completions.create(
        model=OPENROUTER_MODEL,
        messages=[
            {
                "role": "user",
                "content": PROMPT_FACTS_ONLY,
            }
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "payment_facts",
                "strict": True,
                "schema": PaymentFacts.model_json_schema(),
            },
        },
        temperature=0.2,
        max_tokens=4000,
        extra_body={
            "provider": {
                "require_parameters": True
            }
        },
    )

    return PaymentFacts.model_validate_json(
        response.choices[0].message.content
    )

Бизнес-логика в коде:

In [ ]:
def apply_business_logic(facts):
    # Edge case:
    # duplicate_charge имеет приоритет над refund.
    if facts.duplicate_charge:
        category = "duplicate_charge"

    elif facts.refund_requested:
        category = "refund"

    elif facts.payment_failed:
        category = "payment_failed"

    else:
        category = "other"

    urgent = facts.amount >= 1000 or facts.service_unavailable

    return PaymentCase(
        category=category,
        amount=facts.amount,
        urgent=urgent,
        summary=facts.summary,
    )

Запускаем оба варианта:

In [ ]:
prompt_result = classify_with_prompt_logic()

facts = extract_payment_facts()
code_result = apply_business_logic(facts)

In [ ]:
print("Вся логика в prompt:")
print(prompt_result.model_dump())

print("\nИзвлечённые факты:")
print(facts.model_dump())

print("\nЛогика в коде:")
print(code_result.model_dump())

Вся логика в prompt:
{'category': 'duplicate_charge', 'amount': 2400.0, 'urgent': True, 'summary': "Пользователь сообщает о двойном списании платежа в размере 1200 руб. каждый (общая сумма 2400 руб.). Согласно правилам, при наличии одновременной дублирования и запроса на возврат приоритет отдается категории 'duplicate_charge'. Сумма проблемы (2400 руб.) превышает порог в 1000 руб., поэтому статус urgent равен true."}

Извлечённые факты:
{'duplicate_charge': True, 'refund_requested': True, 'payment_failed': False, 'amount': 1200.0, 'service_unavailable': False, 'summary': 'Пользователь сообщает о двойном списании 1200 рублей за одну подписку и требует возврата лишнего суммы.'}

Логика в коде:
{'category': 'duplicate_charge', 'amount': 1200.0, 'urgent': True, 'summary': 'Пользователь сообщает о двойном списании 1200 рублей за одну подписку и требует возврата лишнего суммы.'}


Проверяем эталон. Для этого кейса правильный результат заранее известен:

In [ ]:
expected = {
    "category": "duplicate_charge",
    "amount": 1200.0,
    "urgent": True,
}

In [ ]:
def check_quality(result):
    return {
        "category_ok": result.category == expected["category"],
        "amount_ok": result.amount == expected["amount"],
        "urgent_ok": result.urgent == expected["urgent"],
    }


print("Prompt logic:")
print(check_quality(prompt_result))

print("\nCode logic:")
print(check_quality(code_result))

Prompt logic:
{'category_ok': True, 'amount_ok': False, 'urgent_ok': True}

Code logic:
{'category_ok': True, 'amount_ok': True, 'urgent_ok': True}


### Анализ и вывод

Оба подхода правильно определили категорию `duplicate_charge` и признак `urgent=True`. Однако вариант, где вся бизнес-логика находилась в prompt, неверно интерпретировал поле `amount` как общую сумму двух списаний — `2400`, хотя по условию ожидалось значение одного списания `1200`.

При переносе детерминированных правил в код модель только извлекла факты, после чего Python однозначно применил бизнес-логику. В результате все проверяемые поля совпали с эталоном.

Итог: JSON Schema удобно использовать для структуры и типов, prompt — для интерпретации текста, а однозначные бизнес-правила и edge cases надёжнее реализовывать в коде. Такой подход уменьшает риск неоднозначной трактовки моделью.


### Общий вывод

Эксперименты показали, что Structured Output через JSON Schema заметно надёжнее обычной текстовой инструкции вернуть JSON: схема лучше контролирует структуру и типы и упрощает последующую валидацию через Pydantic.

OpenRouter и GigaChat в протестированных сценариях успешно поддерживали строгий Structured Output. При этом сама схема отвечает прежде всего за форму ответа, а не за корректность бизнес-смысла: правила интерпретации нужно явно задавать в prompt, а детерминированную бизнес-логику и edge cases надёжнее переносить в код.

Итоговый подход для прикладных систем: JSON Schema — для структуры, prompt — для интерпретации текста, Pydantic — для валидации, а код — для однозначных бизнес-правил и fallback-логики.